# Testing ORES

In [10]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Pilot2/1.0 (student research)"}

test_cases = [
    ("Euchambersia",     "en"),
    ("Apple Inc.",       "en"),
    ("Angela Merkel",    "de"),
    ("Berlin",           "de"),
    ("Quantenmechanik",  "de"),
]

for title, lang in test_cases:
    # Correct endpoint — language agnostic outlink model
    url = "https://api.wikimedia.org/service/lw/inference/v1/models/outlink-topic-model:predict"
    
    r = requests.post(url,
        headers={**HEADERS, "Content-Type": "application/json"},
        json={"page_title": title, "lang": lang},
        timeout=15
    )
    
    print(f"\n{title} ({lang}) — status {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        # Response structure may vary — print raw first
        print(f"  {str(data)[:300]}")
    else:
        print(f"  {r.text[:200]}")


Euchambersia (en) — status 200
  {'prediction': {'article': 'https://en.wikipedia.org/wiki/Euchambersia', 'results': [{'topic': 'STEM.STEM*', 'score': 0.960371196269989}, {'topic': 'STEM.Earth_and_environment', 'score': 0.880807101726532}]}}

Apple Inc. (en) — status 200
  {'prediction': {'article': 'https://en.wikipedia.org/wiki/Apple Inc.', 'results': [{'topic': 'History_and_Society.Business_and_economics', 'score': 0.7122421860694885}, {'topic': 'STEM.STEM*', 'score': 0.6297846436500549}]}}

Angela Merkel (de) — status 200
  {'prediction': {'article': 'https://de.wikipedia.org/wiki/Angela Merkel', 'results': [{'topic': 'Geography.Regions.Europe.Europe*', 'score': 0.8354935646057129}, {'topic': 'Culture.Biography.Biography*', 'score': 0.7122421860694885}, {'topic': 'Geography.Regions.Europe.Western_Europe', 'score': 0.61

Berlin (de) — status 200
  {'prediction': {'article': 'https://de.wikipedia.org/wiki/Berlin', 'results': [{'topic': 'Geography.Regions.Europe.Western_Europe', 'sco

# Testing Lift Wing

In [11]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Pilot2/1.0 (student research)"}

def get_rev_id(title, lang):
    api = f"https://{lang}.wikipedia.org/w/api.php"
    r = requests.get(api, headers=HEADERS, params={
        "action": "query", "titles": title,
        "prop": "revisions", "rvprop": "ids",
        "rvlimit": 1, "format": "json"
    }, timeout=10)
    page = list(r.json()["query"]["pages"].values())[0]
    return page["revisions"][0]["revid"] if "revisions" in page else None

test_cases = [
    ("Euchambersia",    "en", "enwiki"),
    ("Apple Inc.",      "en", "enwiki"),
    ("Angela Merkel",   "de", "dewiki"),
    ("Berlin",          "de", "dewiki"),
    ("Quantenmechanik", "de", "dewiki"),
]

print("=== ORES articletopic test ===\n")
for title, lang, wiki in test_cases:
    rev_id = get_rev_id(title, lang)
    print(f"{title} ({wiki}) — rev_id: {rev_id}")
    
    ores_url = f"https://ores.wikimedia.org/v3/scores/{wiki}/{rev_id}/articletopic"
    r = requests.get(ores_url, headers=HEADERS, timeout=10)
    print(f"  ORES status: {r.status_code}")
    
    if r.status_code == 200:
        scores = r.json()[wiki]["scores"][str(rev_id)]["articletopic"]["score"]["probability"]
        top3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
        for topic, prob in top3:
            print(f"  {topic}: {prob:.3f}")
    else:
        print(f"  Response: {r.text[:200]}")
    print()

=== ORES articletopic test ===

Euchambersia (enwiki) — rev_id: 1351502171
  ORES status: 200
  STEM.STEM*: 0.995
  STEM.Earth and environment: 0.993
  STEM.Biology: 0.961

Apple Inc. (enwiki) — rev_id: 1351700241
  ORES status: 200
  Culture.Media.Software: 0.996
  STEM.STEM*: 0.993
  STEM.Technology: 0.991

Angela Merkel (dewiki) — rev_id: 266443579
  ORES status: 400
  Response: {"detail":{"error":{"code":"not found","message":"Models ('articletopic',) not available for dewiki"}}}

Berlin (dewiki) — rev_id: 266611036
  ORES status: 400
  Response: {"detail":{"error":{"code":"not found","message":"Models ('articletopic',) not available for dewiki"}}}

Quantenmechanik (dewiki) — rev_id: 266344648
  ORES status: 400
  Response: {"detail":{"error":{"code":"not found","message":"Models ('articletopic',) not available for dewiki"}}}



# Wikipedia Pilot Study — Iteration 2
### Fixes applied from Iteration 1 results
**Changes from v1:**
- ✅ Stable source switched to Good Articles only (no Featured Articles)
- ✅ Word count cap: stable articles max 8,000 words
- ✅ Matching tolerance tightened: ±20% word count, max 730 days age gap
- ✅ Max match distance cutoff: 5.0 (bad pairs rejected)
- ✅ Dropped: Causal Connectives, Lexical Diversity MTLD
- ✅ Added: Unique Editors, Article Age as model features
- ✅ Talk Page signals moved from validation → model features
- ✅ All saves use `v2_` prefix — v1 files untouched
- ✅ Aggressive save every 5 articles — safe to interrupt anytime


## 0. Setup & Imports

In [24]:
# !pip install requests pytrends lexicalrichness scikit-learn pandas numpy matplotlib seaborn spacy
# !python -m spacy download en_core_web_sm

import requests, time, json, re, random, os
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix
from scipy.stats import mannwhitneyu, norm

try:
    from lexicalrichness import LexicalRichness
    LEXRICH_AVAILABLE = True
    print("✅ lexicalrichness loaded")
except:
    LEXRICH_AVAILABLE = False
    print("⚠️  lexicalrichness not available")

try:
    from pytrends.request import TrendReq
    TRENDS_AVAILABLE = True
    print("✅ pytrends loaded")
except:
    TRENDS_AVAILABLE = False
    print("⚠️  pytrends not available — matching on age + word count only")

HEADERS = {"User-Agent": "UniMannheim-SMDA-Pilot2/1.0 (student research)"}
random.seed(42)
print("\n✅ Ready")


✅ lexicalrichness loaded
✅ pytrends loaded

✅ Ready


## 1. Configuration — v2 settings

In [25]:
V2_CONTESTED_FILE  = "v2_contested.json"
V2_STABLE_FILE     = "v2_stable_pool.json"
V2_PAIRS_FILE      = "v2_matched_pairs.json"
V2_FEATURES_FILE   = "v2_features.csv"
V2_TRENDS_C_CACHE  = "v2_trends_cache_contested.json"
V2_TRENDS_S_CACHE  = "v2_trends_cache_stable.json"

# ── Scale ────────────────────────────────────────────────────────
PILOT_CONTESTED   = 50
PILOT_STABLE_POOL = 300

# ── Article quality thresholds (same as v1) ───────────────────────
MIN_WORDS    = 1000
MIN_EDITORS  = 10
MIN_AGE_DAYS = 365

# ── NEW: Stable word count cap ────────────────────────────────────
MIN_WORDS_STABLE   = 2000
MIN_EDITORS_STABLE = 25 
MAX_WORDS_STABLE = 8000   # v1 had no cap — caused 43% word count gap

# ── NEW: Matching constraints ─────────────────────────────────────
MAX_MATCH_DISTANCE = 5.0  # v1 had pairs with distance 58! — now rejected
MAX_AGE_GAP_DAYS   = 730  # v1 had 1000+ day age gaps — now max 2 years
WORD_COUNT_TOL     = 0.20 # v1 was 0.30 — tightened to 20%

# ── Templates ────────────────────────────────────────────────────
EN_TEMPLATES = ["Template:POV", "Template:Neutrality"]
VALID_YEARS  = [2022, 2023, 2024, 2025]

# ── Topic map ────────────────────────────────────────────────────
TOPIC_BROAD = {
    "STEM":                "science",
    "Culture":             "culture",
    "Geography":           "geography",
    "History_and_Society": "politics_history",
}


print("✅ Config loaded")
print(f"  Max stable word count : {MAX_WORDS_STABLE:,}")
print(f"  Max match distance    : {MAX_MATCH_DISTANCE}")
print(f"  Max age gap (days)    : {MAX_AGE_GAP_DAYS}")
print(f"  Word count tolerance  : ±{int(WORD_COUNT_TOL*100)}%")


✅ Config loaded
  Max stable word count : 8,000
  Max match distance    : 5.0
  Max age gap (days)    : 730
  Word count tolerance  : ±20%


## 2. Helper Functions

In [33]:
def safe_get(url, params, retries=4, base_wait=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if r.status_code == 429:
                print("    🚦 Rate limited — waiting 90s...")
                time.sleep(90)
                continue
            if r.status_code == 200 and r.text.strip():
                return r.json()
            time.sleep(base_wait * (attempt + 1))
        except Exception as e:
            print(f"    ⚠️  {e}")
            time.sleep(base_wait * (attempt + 1))
    return None


def save_json(data, path):
    """Save JSON with a temp file so partial writes never corrupt data."""
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)  # atomic replace


def load_json(path, default=None):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return default if default is not None else []


def is_recent_dispute(text):
    templates = re.findall(
        r'\{\{(?:POV|Neutrality)[^}]*\}\}', text, re.IGNORECASE)
    for t in templates:
        m = re.search(r'date=\w+ (20\d{2})', t)
        if m and int(m.group(1)) in VALID_YEARS:
            return True
    return False


def clean_wikitext(text):
    text = re.sub(r'\{\{[^{}]*\}\}', '', text)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]*)\]\]', r'\1', text)
    text = re.sub(r'==+[^=]+=+', '', text)
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r"'{2,}", '', text)
    return re.sub(r'\s+', ' ', text).strip()


def get_topic_liftwing(title, lang, retries=3):
    url = "https://api.wikimedia.org/service/lw/inference/v1/models/outlink-topic-model:predict"
    
    for attempt in range(retries):
        try:
            r = requests.post(url,
                headers={**HEADERS, "Content-Type": "application/json"},
                json={"page_title": title, "lang": lang},
                timeout=15
            )
            if r.status_code == 200:
                data = r.json()
                
                # DEBUG — print raw response structure
                print(f"    DEBUG raw keys: {list(data.keys())}")
                print(f"    DEBUG prediction keys: {list(data['prediction'].keys())}")
                results = data["prediction"]["results"]
                print(f"    DEBUG first result: {results[0] if results else 'EMPTY'}")
                
                if not results:
                    return "other", None
                top = max(results, key=lambda x: x["score"])
                if top["score"] < 0.5:
                    return "other", top["topic"]
                specific = top["topic"]
                prefix = specific.split(".")[0]
                broad = TOPIC_BROAD.get(prefix, "other")
                return broad, specific
            else:
                print(f"    DEBUG non-200 status: {r.status_code} — {r.text[:100]}")
            time.sleep(2 * (attempt + 1))
        except Exception as e:
            print(f"    DEBUG exception: {e}")
            time.sleep(2 * (attempt + 1))
    
    return "other", None

def get_trends(title, cache):
    if not TRENDS_AVAILABLE:
        return None
    if title in cache:
        return cache[title]
    try:
        pt = TrendReq(hl='en-US', tz=360)
        pt.build_payload([title], timeframe='today 12-m', geo='')
        df = pt.interest_over_time()
        val = round(df[title].mean(), 1) if not df.empty and title in df else None
        cache[title] = val
        return val
    except:
        cache[title] = None
        return None

print("✅ Helpers ready")


✅ Helpers ready


## 3. Article Fetch Function

In [34]:
def fetch_article(title, label, max_words=None, lang="en"):
    """
    Fetch full article data.
    label=0 → contested (applies date filter)
    label=1 → stable (applies max_words cap if set)
    Returns dict or None if article fails any filter.
    """
    url = "https://en.wikipedia.org/w/api.php"

    # ── Text + categories ────────────────────────────────────────
    d1 = safe_get(url, {
        "action":"query", "titles":title,
        "prop":"revisions|categories",
        "rvprop":"content|timestamp", "rvslots":"main",
        "rvlimit":1, "cllimit":50, "format":"json"
    })
    if not d1: return None
    page = list(d1["query"]["pages"].values())[0]
    if "revisions" not in page: return None

    raw  = page["revisions"][0]["slots"]["main"]["*"]
    wc   = len(raw.split())

    if label == 1:
        if wc < MIN_WORDS_STABLE: return None   # stricter for stable
    else:
        if wc < MIN_WORDS: return None          # standard for contested
    if max_words and wc > max_words: return None       # NEW cap for stable
    if label == 0 and not is_recent_dispute(raw): return None

    cats  = [c["title"].replace("Category:","")
             for c in page.get("categories",[])]
    broad_topic, specific_topic = get_topic_liftwing(title, lang)
    time.sleep(1.0)
    clean = clean_wikitext(raw)
    citations = len(re.findall(r'<ref', raw, re.IGNORECASE))
    sections  = len(re.findall(r'^==+[^=]', raw, re.MULTILINE))

    time.sleep(1.2)

    # ── Edit history ─────────────────────────────────────────────
    d2 = safe_get(url, {
        "action":"query", "titles":title,
        "prop":"revisions", "rvprop":"user|timestamp",
        "rvlimit":500, "format":"json"
    })
    revs = []
    if d2:
        revs = list(d2["query"]["pages"].values())[0].get("revisions",[])

    unique_eds  = len(set(r.get("user","") for r in revs))
    total_edits = len(revs)
    if label == 1:
        if unique_eds < MIN_EDITORS_STABLE: return None  # stricter for stable
    else:
        if unique_eds < MIN_EDITORS: return None         # standard for contested

    first = revs[-1]["timestamp"] if revs else None
    age_days = 0
    if first:
        age_days = (datetime.now(timezone.utc) -
                    datetime.fromisoformat(first.replace("Z","+00:00"))).days
        if age_days < MIN_AGE_DAYS: return None

    cutoff = datetime(2025, 10, 1, tzinfo=timezone.utc)
    recent = sum(1 for r in revs
                 if datetime.fromisoformat(
                     r["timestamp"].replace("Z","+00:00")) >= cutoff)
    recency = round(recent / total_edits, 3) if total_edits else 0

    time.sleep(1.2)

    # ── Talk page ────────────────────────────────────────────────
    d3 = safe_get(url, {
        "action":"query", "titles":f"Talk:{title}",
        "prop":"revisions", "rvprop":"comment|content|user",
        "rvslots":"main", "rvlimit":500, "format":"json"
    })
    revert_count = talk_words = talk_editors = 0
    if d3:
        tp   = list(d3["query"]["pages"].values())[0]
        trevs = tp.get("revisions",[])
        revert_count = sum(1 for r in trevs
                           if any(w in r.get("comment","").lower()
                                  for w in ["revert","reverted","undid","undo","restored"]))
        talk_editors = len(set(r.get("user","") for r in trevs))
        if trevs and "slots" in trevs[0]:
            talk_words = len(trevs[0]["slots"]["main"].get("*","").split())

    return {
        "title":title, "label":label,
        "label_name":"contested" if label==0 else "stable",
        "raw_text":raw, "clean_text":clean,
        "word_count":wc, "topic":topic,
        "citation_count":citations, "section_count":sections,
        "unique_editors":unique_eds, "total_edits":total_edits,
        "age_days":age_days, "recency_ratio":recency,
        "revert_count":revert_count,
        "talk_words":talk_words,
        "talk_editors":talk_editors,
        "collected_at":datetime.now().isoformat(),
    }

print("✅ fetch_article ready")


✅ fetch_article ready


## 4. Collect Contested Articles
Resumes automatically if interrupted — saves every 5 articles.

In [35]:
# titles of the articles may change overtime, need to consider this
def fetch_template_titles(template, limit=500):
     # ── Load from cache if available ──────────────────────────────
    cache_file = f"template_titles_{template.replace(':', '_').replace('/', '_')}.json"
    if os.path.exists(cache_file):
        with open(cache_file, encoding="utf-8") as f:
            titles = json.load(f)
        print(f"  ✅ Loaded {len(titles)} titles from cache: {cache_file}")
        return titles
    
    # ── Otherwise fetch from API ───────────────────────────────────
    url = "https://en.wikipedia.org/w/api.php"
    titles, params = [], {
        "action":"query","list":"embeddedin",
        "eititle":template,"eilimit":500,
        "einamespace":0,"format":"json"
    }
    while len(titles) < limit:
        data = safe_get(url, params)
        if not data: break
        titles.extend(p["title"] for p in data["query"]["embeddedin"])
        if "continue" not in data: break
        params["eicontinue"] = data["continue"]["eicontinue"]
        time.sleep(1)
    return titles

# ── Load existing progress ────────────────────────────────────────
contested = load_json(V2_CONTESTED_FILE, [])
print(f"Fixing topics for {len(contested)} contested articles...")

for art in contested:
    broad, specific = get_topic_liftwing(art["title"], "en")
    art["topic"]          = broad
    art["topic_specific"] = specific
    print(f"  {art['title'][:40]} → {broad} ({specific})")
    time.sleep(2.0)

save_json(contested, V2_CONTESTED_FILE)
print("✅ Done")

seen_c    = set(a["title"] for a in contested)
have      = len(contested)
need      = PILOT_CONTESTED - have

print(f"Contested: have {have}, need {need} more")

if need > 0:
    all_titles = []
    for t in EN_TEMPLATES:
        titles = fetch_template_titles(t, limit=400)
        all_titles.extend(titles)
        print(f"  {t}: {len(titles)} titles")
    
    random.shuffle(all_titles)
    all_titles = [t for t in all_titles if t not in seen_c]
    print(f"  {len(all_titles)} unique candidates remaining")

    for title in all_titles:
        if len(contested) >= PILOT_CONTESTED: break
        time.sleep(random.uniform(2.5, 4.0))
        art = fetch_article(title, label=0)
        if art:
            contested.append(art)
            seen_c.add(title)
            print(f"  [{len(contested)}/{PILOT_CONTESTED}] ✅ {title[:50]} "
                  f"({art['word_count']}w, {art['age_days']}d, {art['topic']})")
            if len(contested) % 5 == 0:          # save every 5
                save_json(contested, V2_CONTESTED_FILE)
                print(f"  💾 Saved at {len(contested)}")
        else:
            print(f"  ⊘ {title[:50]}")
    
    save_json(contested, V2_CONTESTED_FILE)

print(f"\n✅ Contested total: {len(contested)}/{PILOT_CONTESTED}")


Fixing topics for 50 contested articles...
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'Geography.Regions.Africa.Africa*', 'score': 0.9783946871757507}
  Kavirondo → geography (Geography.Regions.Africa.Africa*)
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'STEM.STEM*', 'score': 0.7663036584854126}
  Paul Sereno → science (STEM.STEM*)
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'Culture.Media.Music', 'score': 0.9926641583442688}
  Manele → culture (Culture.Media.Music)
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'History_and_Society.Politics_and_government', 'score': 0.5544804334640503}
  1999 Seattle WTO protests → politics_history (History_and_Society.Politics_and_government)
    DEBUG raw k

## 5. Collect Stable Articles
Good Articles only. Max 8,000 words. Saves every 5 articles.

In [36]:
def has_dispute_history(title):
    """Check last 50 revisions for any dispute template."""
    url  = "https://en.wikipedia.org/w/api.php"
    pat  = re.compile(r'\{\{(?:POV|Disputed|Neutrality|Biased|Unbalanced)', re.IGNORECASE)
    data = safe_get(url, {
        "action":"query","titles":title,
        "prop":"revisions","rvprop":"content",
        "rvslots":"main","rvlimit":50,"format":"json"
    })
    if not data: return False
    page = list(data["query"]["pages"].values())[0]
    for rv in page.get("revisions",[]):
        if pat.search(rv.get("slots",{}).get("main",{}).get("*","")):
            return True
    return False


def fetch_good_article_titles(n=600):
    """Fetch ALL Good Article titles, shuffle, then return n."""
    # ── Load from saved file if it exists — skip the whole API fetch ──
    if os.path.exists("good_article_titles_en.json"):
        with open("good_article_titles_en.json", encoding="utf-8") as f:
            titles = json.load(f)
        random.shuffle(titles)
        print(f"  ✅ Loaded {len(titles)} titles from saved file (shuffled)")
        print(f"  First 5: {titles[:5]}")
        return titles
    

    url    = "https://en.wikipedia.org/w/api.php"
    titles = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Good articles",
        "cmlimit": 500,
        "cmtype": "page",
        "cmnamespace": 0,
        "format": "json"
    }
    print("  Fetching ALL Good Article titles first...")
    while True:
        data = safe_get(url, params)
        if not data: break
        batch = [p["title"] for p in data["query"]["categorymembers"]
                 if not p["title"].startswith("Talk:")]
        titles.extend(batch)
        print(f"  ... {len(titles)} titles fetched so far")
        if "continue" not in data: break
        params["cmcontinue"] = data["continue"]["cmcontinue"]
        time.sleep(1)
    
    print(f"  Total: {len(titles)} Good Article titles — shuffling...")
    random.shuffle(titles)
    with open("good_article_titles_en.json", "w", encoding="utf-8") as f:
        json.dump(titles, f, ensure_ascii=False)
    print(f"  💾 Titles saved to good_article_titles_en.json")
    
    print(f"  First 5 after shuffle: {titles[:5]}")
    return titles

In [37]:
# import requests

# HEADERS = {"User-Agent": "UniMannheim-SMDA-Pilot2/1.0 (student research)"}

# test_articles = [
#     ("2011 Veranda's Willems–Accent season", "en"),  # cycling — got STEM.Technology
#     ("Noble train of artillery",              "en"),  # military history — got STEM.Technology
#     ("Apple River Fort",                      "en"),  # historical fort — got STEM.Technology
#     ("Steve Longa",                           "en"),  # person? — got STEM.Technology
#     ("SMS Stettin",                           "en"),  # warship — got STEM.Technology
#     # Known correct ones for comparison
#     ("Euchambersia",                          "en"),  # should be STEM.Biology
#     ("Angela Merkel",                         "de"),  # should be Culture/Geography
# ]

# url = "https://api.wikimedia.org/service/lw/inference/v1/models/outlink-topic-model:predict"

# for title, lang in test_articles:
#     r = requests.post(url,
#         headers={**HEADERS, "Content-Type": "application/json"},
#         json={"page_title": title, "lang": lang},
#         timeout=15
#     )
#     print(f"\n{title} ({lang}) — status {r.status_code}")
#     if r.status_code == 200:
#         results = r.json()["prediction"]["results"]
#         if results:
#             top3 = sorted(results, key=lambda x: x["score"], reverse=True)[:3]
#             for t in top3:
#                 print(f"  {t['topic']}: {t['score']:.3f}")
#         else:
#             print("  No results returned")
#     else:
#         print(f"  {r.text[:150]}")

In [38]:

# ── Load existing progress ────────────────────────────────────────
stable = load_json(V2_STABLE_FILE, [])
print(f"Fixing topics for {len(stable)} stable articles...")

for art in stable:
    broad, specific = get_topic_liftwing(art["title"], "en")
    art["topic"]          = broad
    art["topic_specific"] = specific
    print(f"  {art['title'][:40]} → {broad} ({specific})")
    time.sleep(2.0)

save_json(stable, V2_STABLE_FILE)
print("✅ Done")

seen_s = set(a["title"] for a in stable)
have_s = len(stable)
need_s = PILOT_STABLE_POOL - have_s

print(f"Stable pool: have {have_s}, need {need_s} more")

if need_s > 0:
    candidates = fetch_good_article_titles(n=PILOT_STABLE_POOL * 4)
    candidates = [t for t in candidates if t not in seen_s]
    print(f"  {len(candidates)} new candidates to try\n")
    
    for title in candidates:
        if len(stable) >= PILOT_STABLE_POOL: break
        time.sleep(random.uniform(2.5, 4.0))
        
        if has_dispute_history(title):
            print(f"  ⊘ {title[:50]} — dispute in history")
            continue
        
        art = fetch_article(title, label=1, max_words=MAX_WORDS_STABLE)
        if art:
            stable.append(art)
            seen_s.add(title)
            print(f"  [{len(stable)}/{PILOT_STABLE_POOL}] ✅ {title[:50]} "
                  f"({art['word_count']}w, {art['topic']})")
            if len(stable) % 5 == 0:
                save_json(stable, V2_STABLE_FILE)
                print(f"  💾 Saved at {len(stable)}")
        else:
            print(f"  ⊘ {title[:50]}")
    
    save_json(stable, V2_STABLE_FILE)

print(f"\n✅ Stable pool total: {len(stable)}/{PILOT_STABLE_POOL}")

Fixing topics for 10 stable articles...
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'Culture.Media.Media*', 'score': 0.9648651480674744}
  Cyprus in the Eurovision Song Contest 20 → culture (Culture.Media.Media*)
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'Culture.Internet_culture', 'score': 0.982567548751831}
  VanossGaming → culture (Culture.Internet_culture)
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'Culture.Media.Media*', 'score': 0.9724247455596924}
  Dust My Broom → culture (Culture.Media.Media*)
    DEBUG raw keys: ['prediction']
    DEBUG prediction keys: ['article', 'results']
    DEBUG first result: {'topic': 'Culture.Media.Media*', 'score': 0.9958112239837646}
  A Flash Flood of Colour → culture (Culture.Media.Media*)
    DEBUG raw keys: ['prediction'

KeyboardInterrupt: 

## 6. Google Trends Scores
Cached separately — safe to interrupt and resume.

In [ ]:
trends_c = load_json(V2_TRENDS_C_CACHE, {})
trends_s = load_json(V2_TRENDS_S_CACHE, {})

def enrich_trends(articles, cache, cache_file):
    missing = [a for a in articles if a["title"] not in cache]
    print(f"  Fetching trends for {len(missing)} articles...")
    for i, art in enumerate(missing):
        score = get_trends(art["title"], cache)
        if (i+1) % 10 == 0:
            save_json(cache, cache_file)
            print(f"  💾 Trends cache saved ({i+1}/{len(missing)})")
        time.sleep(random.uniform(3, 5))
    save_json(cache, cache_file)
    for art in articles:
        art["trends_avg"] = cache.get(art["title"])
    return articles

if TRENDS_AVAILABLE:
    print("Enriching contested with trends...")
    contested = enrich_trends(contested, trends_c, V2_TRENDS_C_CACHE)
    print("Enriching stable with trends...")
    stable    = enrich_trends(stable, trends_s, V2_TRENDS_S_CACHE)
    print("✅ Trends done")
else:
    for a in contested + stable:
        a["trends_avg"] = None
    print("⚠️  Skipping trends — pytrends not available")


## 7. Matching — v2 Improvements
Three fixes from v1:
- Max distance cutoff: 5.0 (rejects bad pairs)
- Max age gap: 730 days (rejects age-mismatched pairs)  
- Word count tolerance: ±20% (tighter than v1's ±30%)


In [ ]:
def match_v2(contested, stable_pool):
    """
    Improved nearest-neighbour matching with hard rejection rules.
    Each stable article can only be matched once.
    Pairs that exceed distance or age/word thresholds are REJECTED.
    """
    used  = set()
    pairs = []
    rejected = []

    def feats(a):
        return np.array([
            a.get("word_count", 0) / 10000,
            a.get("age_days",   0) / 3650,
            (a.get("trends_avg") or 0) / 100,
        ])

    stable_list  = list(stable_pool)
    stable_feats = np.array([feats(a) for a in stable_list])
    nn = NearestNeighbors(
        n_neighbors=min(30, len(stable_list)), metric='euclidean')
    nn.fit(stable_feats)

    for c in contested:
        dists, idxs = nn.kneighbors(feats(c).reshape(1,-1))
        matched = None

        for dist, idx in zip(dists[0], idxs[0]):
            if dist > MAX_MATCH_DISTANCE:       # NEW: reject bad distance
                break
            if idx in used:
                continue
            s = stable_list[idx]

            # NEW: hard age gap check
            if abs(c["age_days"] - s["age_days"]) > MAX_AGE_GAP_DAYS:
                continue

            # NEW: tighter word count check
            wc_ratio = abs(c["word_count"] - s["word_count"]) / max(c["word_count"], 1)
            if wc_ratio > WORD_COUNT_TOL:
                continue

            matched = (s, idx, dist)
            if s["topic"] == c["topic"]:
                break   # perfect topic match — stop searching

        if matched:
            s, idx, dist = matched
            used.add(idx)
            pairs.append({
                "contested": c, "stable": s,
                "match_distance": round(dist, 4),
                "same_topic": c["topic"] == s["topic"],
                "age_gap":    abs(c["age_days"] - s["age_days"]),
                "wc_gap_pct": round(abs(c["word_count"]-s["word_count"])
                                    / max(c["word_count"],1) * 100, 1),
            })
        else:
            rejected.append(c["title"])

    return pairs, rejected

print("Running v2 matching...")
pairs_v2, rejected = match_v2(contested, stable)

print(f"\n✅ Matched:  {len(pairs_v2)} pairs")
print(f"⊘  Rejected: {len(rejected)} contested articles (no valid match found)")
if rejected:
    print("  Rejected titles:")
    for t in rejected:
        print(f"    - {t}")

# Summary
same_topic = sum(1 for p in pairs_v2 if p["same_topic"])
avg_dist   = np.mean([p["match_distance"] for p in pairs_v2])
avg_age_gap = np.mean([p["age_gap"] for p in pairs_v2])
avg_wc_gap  = np.mean([p["wc_gap_pct"] for p in pairs_v2])

print(f"\n  Same topic    : {same_topic}/{len(pairs_v2)} ({100*same_topic//max(len(pairs_v2),1)}%)")
print(f"  Avg distance  : {avg_dist:.4f}  (v1 was 5.39)")
print(f"  Avg age gap   : {avg_age_gap:.0f} days  (v1 was 1017 days)")
print(f"  Avg WC gap    : {avg_wc_gap:.1f}%  (v1 was 43%)")

save_json([{
    "contested": p["contested"], "stable": p["stable"],
    "match_distance": p["match_distance"], "same_topic": p["same_topic"],
    "age_gap": p["age_gap"], "wc_gap_pct": p["wc_gap_pct"]
} for p in pairs_v2], V2_PAIRS_FILE)
print(f"\n✅ Saved to {V2_PAIRS_FILE}")


### 7.1 Matching Quality Check

In [ ]:
print(f"{'CONTESTED':<38} {'STABLE':<38} {'DIST':>6}  {'AGE Δ':>6}  {'WC Δ%':>5}  TOPIC")
print("-" * 105)
for p in pairs_v2[:15]:
    c, s = p["contested"], p["stable"]
    tf = "✅" if p["same_topic"] else "⚠️ "
    print(f"{c['title'][:36]:<38} {s['title'][:36]:<38} "
          f"{p['match_distance']:>6.3f}  {p['age_gap']:>6.0f}  "
          f"{p['wc_gap_pct']:>5.1f}%  {tf} {c['topic']} → {s['topic']}")

# Distribution plots
c_ages  = [p["contested"]["age_days"]   for p in pairs_v2]
s_ages  = [p["stable"]["age_days"]      for p in pairs_v2]
c_words = [p["contested"]["word_count"] for p in pairs_v2]
s_words = [p["stable"]["word_count"]    for p in pairs_v2]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("v2 Matching Quality (should be much tighter than v1)",
             fontsize=13, fontweight='bold')

for ax, (c_vals, s_vals, xlabel) in zip(axes, [
    (c_ages,  s_ages,  "Article Age (days)"),
    (c_words, s_words, "Word Count"),
]):
    ax.hist(c_vals, bins=20, alpha=0.6, color='#DC2626', label='Contested', density=True)
    ax.hist(s_vals, bins=20, alpha=0.6, color='#059669', label='Stable',    density=True)
    ax.set_xlabel(xlabel)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

axes[0].set_title(f"Age: C={np.mean(c_ages):.0f}d vs S={np.mean(s_ages):.0f}d  (v1: 6494 vs 5477)")
axes[1].set_title(f"Words: C={np.mean(c_words):.0f} vs S={np.mean(s_words):.0f}  (v1: 4574 vs 6529)")
plt.tight_layout()
plt.savefig("v2_matching_quality.png", dpi=150, bbox_inches='tight')
plt.show()


## 8. Feature Extraction — v2 Feature Set
**Dropped from v1:** Causal Connectives, Lexical Diversity MTLD  
**Added in v2:** Unique Editors, Article Age  
**Moved from validation → features:** Revert Count, Talk Words, Talk Editors


In [ ]:
HEDGING_WORDS = [
    "allegedly","apparently","arguably","claims","claimed",
    "reportedly","supposedly","some argue","some suggest",
    "it is claimed","it has been argued","according to some",
    "disputed","controversial","contentious","debated",
    "critics say","critics argue","opponents claim",
    "proponents argue","others believe","many believe",
]
DEFINITION_PATTERNS = [
    r'\bis (a|an|the)\b', r'\brefers to\b', r'\bdefined as\b',
    r'\bknown as\b', r'\bdescribed as\b', r'\bconsidered (a|an|the)\b',
]

def extract_features_v2(article):
    text      = article.get("clean_text", "")
    words     = text.lower().split()
    n         = len(words)
    if n == 0: return None
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 10]
    n_sent    = max(len(sentences), 1)
    text_low  = text.lower()

    hedge_count = sum(text_low.count(w) for w in HEDGING_WORDS)
    hedging     = round(hedge_count / n * 1000, 4)

    def_sents = sum(1 for s in sentences
                    if any(re.search(p, s, re.IGNORECASE) for p in DEFINITION_PATTERNS))
    def_ratio = round(def_sents / n_sent, 4)

    return {
        "title":          article["title"],
        "label":          article["label"],
        "label_name":     article["label_name"],
        "topic":          article["topic"],
        "topic_specific": article.get("topic_specific"),
        # Linguistic (kept)
        "hedging_density": hedging,
        "def_ratio":       def_ratio,
        # Structural
        "citation_count":  article.get("citation_count", 0),
        "section_count":   article.get("section_count", 0),
        "unique_editors":  article.get("unique_editors", 0),   # NEW
        "age_days":        article.get("age_days", 0),          # NEW
        "recency_ratio":   article.get("recency_ratio", 0),
        # Talk page — now MODEL features not just validation
        "revert_count":    article.get("revert_count", 0),      # MOVED
        "talk_words":      article.get("talk_words", 0),         # MOVED
        "talk_editors":    article.get("talk_editors", 0),       # MOVED
        # Matching variables (kept for reference, not used in model)
        "word_count":      article.get("word_count", n),
        "trends_avg":      article.get("trends_avg"),
    }

records = []
for p in pairs_v2:
    cf = extract_features_v2(p["contested"])
    sf = extract_features_v2(p["stable"])
    if cf and sf:
        records += [cf, sf]

df = pd.DataFrame(records)
df.to_csv(V2_FEATURES_FILE, index=False)
print(f"✅ Feature matrix: {df.shape[0]} articles × {df.shape[1]} columns")
print(f"   Contested: {(df.label==0).sum()} | Stable: {(df.label==1).sum()}")
df.head(4)


## 9. Exploratory Analysis

In [ ]:
FEATURE_COLS = [
    "hedging_density","def_ratio","citation_count",
    "section_count","unique_editors","age_days",
    "recency_ratio","revert_count","talk_words","talk_editors"
]
LABELS = {
    "hedging_density": "Hedging Density",
    "def_ratio":       "Definition Ratio",
    "citation_count":  "Citation Count",
    "section_count":   "Section Count",
    "unique_editors":  "Unique Editors",
    "age_days":        "Article Age (days)",
    "recency_ratio":   "Edit Recency Ratio",
    "revert_count":    "Revert Count",
    "talk_words":      "Talk Word Count",
    "talk_editors":    "Talk Editors",
}

c_df = df[df.label==0]
s_df = df[df.label==1]

print(f"{'Feature':<25} {'Contested':>12} {'Stable':>12} {'Diff%':>8}  v1 sig?")
print("-"*65)
V1_SIG = {"citation_count","section_count","recency_ratio"}
for col in FEATURE_COLS:
    cm, sm = c_df[col].mean(), s_df[col].mean()
    diff   = (cm-sm)/(sm+1e-9)*100
    sig    = "✅ was sig" if col in V1_SIG else ""
    print(f"{LABELS[col]:<25} {cm:>12.3f} {sm:>12.3f} "
          f"{'↑' if diff>0 else '↓'}{abs(diff):>6.1f}%  {sig}")


In [ ]:
# Violin plots — all features
n_cols = 5
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 9))
fig.suptitle("v2 Feature Distributions: Contested vs Stable",
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    parts = ax.violinplot(
        [c_df[col].dropna().values, s_df[col].dropna().values],
        positions=[0,1], showmedians=True)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#DC2626','#059669'][j])
        pc.set_alpha(0.7)
    ax.set_xticks([0,1])
    ax.set_xticklabels(['Contested','Stable'])
    ax.set_title(LABELS[col], fontsize=10)
    ax.grid(axis='y', alpha=0.3)

for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("v2_feature_distributions.png", dpi=150, bbox_inches='tight')
plt.show()


## 10. Mann-Whitney U Tests

In [ ]:
print("Mann-Whitney U — v2 Results\n")
print(f"{'Feature':<25} {'p-value':>9} {'Effect r':>9}  Direction  Sig?   v1 result")
print("-"*75)

V1_RESULTS = {
    "hedging_density":  ("p=0.173","❌"),
    "def_ratio":        ("p=0.112","❌"),
    "citation_count":   ("p=0.038","✅"),
    "section_count":    ("p=0.050","✅"),
    "recency_ratio":    ("p=0.006","✅"),
    "revert_count":     ("p=0.100","~"),
    "talk_words":       ("p=0.301","❌"),
    "talk_editors":     ("p=0.028","✅ C<S"),
    "unique_editors":   ("NEW","—"),
    "age_days":         ("NEW","—"),
}

mw_results = []
for col in FEATURE_COLS:
    cv = c_df[col].dropna().values
    sv = s_df[col].dropna().values
    if len(cv)<3 or len(sv)<3: continue
    u, p = mannwhitneyu(cv, sv, alternative='two-sided')
    z    = norm.ppf(1-p/2)
    r    = abs(z)/np.sqrt(len(cv)+len(sv))
    dire = "C > S" if cv.mean()>sv.mean() else "C < S"
    sig  = "✅" if p<0.05 else ("~" if p<0.10 else "❌")
    v1p, v1s = V1_RESULTS.get(col, ("—","—"))
    print(f"{LABELS[col]:<25} {p:>9.4f} {r:>9.3f}  {dire}      {sig}    {v1p} {v1s}")
    mw_results.append({"feature":col,"p":p,"r":r,"direction":dire,"sig":p<0.05})

print("\n✅ = p<0.05   ~ = p<0.10   ❌ = not significant")


## 11. Logistic Regression — v2 Model

In [ ]:
MODEL_FEATURES = [
    "hedging_density","def_ratio",
    "citation_count","section_count",
    "unique_editors","age_days","recency_ratio",
    "revert_count","talk_words","talk_editors",
]

df_m   = df[MODEL_FEATURES+["label"]].dropna()
X      = df_m[MODEL_FEATURES].values
y      = df_m["label"].values
scaler = StandardScaler()
Xs     = scaler.fit_transform(X)

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model  = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

f1s    = cross_val_score(model, Xs, y, cv=cv, scoring='f1_macro')
accs   = cross_val_score(model, Xs, y, cv=cv, scoring='accuracy')
base   = max(y.mean(), 1-y.mean())

print("5-fold Cross-Validation — v2")
print(f"  F1 (macro):  {f1s.mean():.3f} ± {f1s.std():.3f}   (v1: 0.676 ± 0.099)")
print(f"  Accuracy:    {accs.mean():.3f} ± {accs.std():.3f}   (v1: 0.680 ± 0.098)")
print(f"  Baseline:    {base:.3f}")
print(f"  Improvement: {accs.mean()-base:+.3f}              (v1: +0.180)")


In [ ]:
model.fit(Xs, y)
coef_df = pd.DataFrame({
    "feature":     [LABELS[f] for f in MODEL_FEATURES],
    "coefficient": model.coef_[0],
}).sort_values("coefficient")

fig, ax = plt.subplots(figsize=(9,6))
colors  = ['#DC2626' if c<0 else '#059669' for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"],
        color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel("Standardised Coefficient (← Contested  |  Stable →)", fontsize=11)
ax.set_title("v2 Logistic Regression Coefficients\n"
             "Dropped: Causal Connectives, MTLD  |  Added: Unique Editors, Age, Talk features",
             fontsize=11, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#DC2626', alpha=0.85, label='Predicts Contested'),
    mpatches.Patch(color='#059669', alpha=0.85, label='Predicts Stable'),
], loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("v2_coefficients.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nCoefficients:")
for _, row in coef_df.iterrows():
    arrow = "← Contested" if row["coefficient"]<0 else "→ Stable"
    print(f"  {row['feature']:<28} {row['coefficient']:>+7.4f}  {arrow}")


## 12. v2 Summary — How Did the Fixes Help?

In [ ]:
print("="*65)
print("ITERATION 2 SUMMARY")
print("="*65)

print(f"\n📊 Dataset")
print(f"   Matched pairs    : {len(pairs_v2)}")
print(f"   Rejected (no match): {len(rejected)}")
print(f"   Same-topic pairs : {same_topic}/{len(pairs_v2)}")

print(f"\n📐 Matching Quality vs v1")
print(f"   Avg distance : {avg_dist:.3f}  (v1: 5.388 — lower is better)")
print(f"   Avg age gap  : {avg_age_gap:.0f} days  (v1: 1017 days)")
print(f"   Avg WC gap   : {avg_wc_gap:.1f}%  (v1: 43%)")

print(f"\n📈 Model Performance vs v1")
print(f"   F1 macro  : {f1s.mean():.3f} ± {f1s.std():.3f}  (v1: 0.676)")
print(f"   Accuracy  : {accs.mean():.3f} ± {accs.std():.3f}  (v1: 0.680)")
print(f"   Baseline  : {base:.3f}")

print(f"\n🔍 Feature Changes")
print("   Dropped : Causal Connectives (p=0.45 in v1), MTLD (p=0.43 in v1)")
print("   Added   : Unique Editors, Article Age (days)")
print("   Moved   : Talk Page signals → model features (from validation only)")

sig = [r for r in mw_results if r["sig"]]
not_sig = [r for r in mw_results if not r["sig"]]
print(f"\n✅ Significant features (p<0.05): {len(sig)}")
for r in sorted(sig, key=lambda x: x["r"], reverse=True):
    print(f"   {LABELS.get(r['feature'],r['feature']):<28} p={r['p']:.3f}  r={r['r']:.3f}  {r['direction']}")
print(f"\n❌ Not significant: {len(not_sig)}")
for r in not_sig:
    print(f"   {LABELS.get(r['feature'],r['feature']):<28} p={r['p']:.3f}")

print(f"\n🚀 Next Steps")
print("   → If F1 improved vs v1: fixes worked — scale to 200 per group")
print("   → If F1 similar: matching quality was not the bottleneck")
print("   → Either way: replicate for German Wikipedia (DE)")
print("   → Cross-lingual coefficient comparison = midterm main finding")
print("="*65)
